# 🍽️ Food Recommendation System — Apriori Training Notebook

This notebook explains **step-by-step how the Apriori recommendation system works**, using the same core logic as the Streamlit project.

### What we will learn
1. Load transaction data
2. Understand transactions
3. Convert transactions into a machine-learning-friendly format
4. Generate frequent itemsets using Apriori
5. Generate association rules
6. Understand **Support, Confidence and Lift**
7. Build a recommendation function
8. Test recommendations
9. Save the learned frequent itemsets and rules
10. Understand how new customer orders are added for future retraining

## 1. Install / Import Libraries

We use:
- `pandas` → read and process transaction data
- `mlxtend.preprocessing.TransactionEncoder` → convert basket transactions into a binary table
- `mlxtend.frequent_patterns.apriori` → find frequent item combinations
- `mlxtend.frequent_patterns.association_rules` → generate recommendation rules

In [1]:
# If needed, run this once:
# !pip install pandas mlxtend openpyxl

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the Transaction Dataset

The project reads `Apriori_500_Realistic_Transactions.csv`.

The important column is **`Items`**.

Each row represents one customer transaction, for example:

`Pizza, Garlic Bread, Coke`

This means one customer bought these three items together.

In [2]:
DATA_FILE = "Apriori_500_Realistic_Transactions.csv"

transaction_df = pd.read_csv(DATA_FILE)

print("Shape:", transaction_df.shape)
display(transaction_df.head(10))

Shape: (5000, 2)


,Transaction_ID,Items
0,1,"Vada Pav, Tea, Milkshake"
1,2,"Hot Dog, Veg Burger, Coke, Chocolate Cake"
2,3,"Ice Cream, Brownie, Sweet Corn"
3,4,"Pizza, Garlic Bread, Pepsi, Lemon Soda"
4,5,"Hot Dog, Veg Burger, Coke"
5,6,"Burger, Fries, Coke"
6,7,"Ice Cream, Brownie"
7,8,"Idli, Sambar, Filter Coffee"
8,9,"Burger, Fries, Coke"
9,10,"Masala Dosa, Filter Coffee"


## 3. Convert Each Row into a Transaction List

Apriori expects transactions like:

```text
[
    ["Pizza", "Coke"],
    ["Burger", "Fries", "Coke"],
    ["Pizza", "Garlic Bread", "Coke"]
]
```

The Streamlit project splits the comma-separated `Items` column and removes extra spaces.

In [3]:
transactions = (
    transaction_df["Items"]
    .fillna("")
    .apply(lambda x: [item.strip() for item in x.split(",") if item.strip()])
    .tolist()
)

print("First 5 transactions:")
for t in transactions[:5]:
    print(t)

First 5 transactions:
['Vada Pav', 'Tea', 'Milkshake']
['Hot Dog', 'Veg Burger', 'Coke', 'Chocolate Cake']
['Ice Cream', 'Brownie', 'Sweet Corn']
['Pizza', 'Garlic Bread', 'Pepsi', 'Lemon Soda']
['Hot Dog', 'Veg Burger', 'Coke']


## 4. Why Do We Need Transaction Encoding?

Apriori cannot directly work with the text list.

We convert transactions into a **One-Hot / Boolean matrix**:

| Transaction | Pizza | Burger | Fries | Coke |
|---|---:|---:|---:|---:|
| T1 | 1 | 0 | 0 | 1 |
| T2 | 0 | 1 | 1 | 1 |
| T3 | 1 | 0 | 0 | 1 |

`1` = item is present in that transaction  
`0` = item is absent

This is exactly the encoding step used in the project with `TransactionEncoder`.

In [4]:
te = TransactionEncoder()

encoded_array = te.fit(transactions).transform(transactions)

encoded_df = pd.DataFrame(
    encoded_array,
    columns=te.columns_
)

print("Encoded shape:", encoded_df.shape)
display(encoded_df.head())

Encoded shape: (5000, 55)


,Biryani,Brownie,Burger,Chocolate Cake,Chole Bhature,Coffee,Coke,Cold Coffee,Curd,Donut,...,Samosa,Sandwich,Soup,Spring Roll,Sprite,Sweet Corn,Tea,Vada Pav,Veg Burger,Veg Fried Rice
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,True,False,False
1,False,False,False,True,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,True,False


## 5. Choose Minimum Support

### Support

Support tells us **how frequently an itemset appears in all transactions**.

Conceptually:

**Support(A) = transactions containing A / total transactions**

Example:

If Pizza appears in 100 out of 500 transactions:

**Support(Pizza) = 100 / 500 = 0.20 = 20%**

Here we use `min_support = 0.02`, matching the current Streamlit default.

Only itemsets meeting this minimum support are kept.

In [5]:
MIN_SUPPORT = 0.02

frequent_itemsets = apriori(
    encoded_df,
    min_support=MIN_SUPPORT,
    use_colnames=True
)

print("Number of frequent itemsets:", len(frequent_itemsets))
display(
    frequent_itemsets
    .sort_values("support", ascending=False)
    .head(20)
)

Number of frequent itemsets: 98


,support,itemsets
44,0.2070,frozenset({Tea})
5,0.2018,frozenset({Coke})
1,0.1064,frozenset({Brownie})
17,0.1018,frozenset({Lassi})
31,0.1002,frozenset({Pepsi})
10,0.0972,frozenset({Garlic Bread})
8,0.0952,frozenset({Filter Coffee})
42,0.0896,frozenset({Sprite})
0,0.0544,frozenset({Biryani})
4,0.0544,frozenset({Coffee})


## 6. What Does Apriori Actually Do?

Apriori works progressively:

### Step A — Find frequent 1-itemsets
Examples:
- `{Pizza}`
- `{Burger}`
- `{Coke}`

### Step B — Find frequent 2-itemsets
Examples:
- `{Pizza, Coke}`
- `{Burger, Fries}`

### Step C — Find frequent 3-itemsets
Examples:
- `{Burger, Fries, Coke}`

At every stage, itemsets below the minimum support are removed.

### Key idea
If an itemset is not frequent, its larger combinations cannot become frequent.

This is the **Apriori principle** and helps reduce unnecessary combinations.

## 7. Generate Association Rules

Frequent itemsets tell us **what items occur together**.

Association rules tell us **what item can be recommended when another item is selected**.

Example:

**{Pizza} → {Coke}**

Meaning:

> Customers who buy Pizza often also buy Coke.

The project uses `association_rules()` with **confidence** as the filtering metric.

In [6]:
MIN_CONFIDENCE = 0.02

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

print("Number of rules:", len(rules))
display(
    rules[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].sort_values("confidence", ascending=False).head(20)
)

Number of rules: 140


,antecedents,consequents,support,confidence,lift
0,frozenset({Biryani}),frozenset({Gulab Jamun}),0.0544,1.0,18.382353
1,frozenset({Gulab Jamun}),frozenset({Biryani}),0.0544,1.0,18.382353
2,frozenset({Biryani}),frozenset({Raita}),0.0544,1.0,18.382353
3,frozenset({Raita}),frozenset({Biryani}),0.0544,1.0,18.382353
4,frozenset({Coffee}),frozenset({Brownie}),0.0544,1.0,9.398496
6,frozenset({Ice Cream}),frozenset({Brownie}),0.0520,1.0,9.398496
10,frozenset({Fries}),frozenset({Burger}),0.0490,1.0,20.408163
8,frozenset({Burger}),frozenset({Coke}),0.0490,1.0,4.955401
11,frozenset({Burger}),frozenset({Fries}),0.0490,1.0,20.408163
18,frozenset({Hot Dog}),frozenset({Coke}),0.0504,1.0,4.955401


## 8. Understand Support, Confidence and Lift

### ① Support
How often the complete item combination appears in the dataset.

### ② Confidence
For a rule:

**Pizza → Coke**

confidence answers:

> Among customers who bought Pizza, how many also bought Coke?

### ③ Lift
Lift compares the rule with how common the recommended item is by itself.

- **Lift > 1** → positive association
- **Lift ≈ 1** → little/no useful association
- **Lift < 1** → negative association

For recommendation systems, looking at all three metrics is better than using confidence alone.

In [7]:
def recommend(item, rules_df, top_n=2):
    if rules_df.empty:
        return pd.DataFrame()

    matching = rules_df[
        rules_df["antecedents"].apply(lambda x: item in x)
    ].copy()

    if matching.empty:
        return pd.DataFrame()

    matching = matching.sort_values(
        by="confidence",
        ascending=False
    )

    return matching.head(top_n)


# Change this to any item available in your dataset
selected_item = transactions[0][0]

result = recommend(selected_item, rules, top_n=5)

print("Selected food:", selected_item)

if result.empty:
    print("No recommendation rule found.")
else:
    display(
        result[
            ["antecedents", "consequents", "support", "confidence", "lift"]
        ]
    )

Selected food: Vada Pav


,antecedents,consequents,support,confidence,lift
79,frozenset({Vada Pav}),frozenset({Tea}),0.0544,1.0,4.830918


In [8]:
def get_recommendation_names(item, rules_df, top_n=2):
    result = recommend(item, rules_df, top_n=top_n)

    recommendations = []

    for _, row in result.iterrows():
        for rec_item in list(row["consequents"]):
            rec_item = str(rec_item).strip()

            if rec_item != item and rec_item not in recommendations:
                recommendations.append(rec_item)

    return recommendations[:top_n]


recommendations = get_recommendation_names(
    selected_item,
    rules,
    top_n=5
)

print("Selected food:", selected_item)
print("Recommended food:", recommendations)

Selected food: Vada Pav
Recommended food: ['Tea']


In [9]:
# Save frequent itemsets
frequent_itemsets_to_save = frequent_itemsets.copy()
frequent_itemsets_to_save["itemsets"] = frequent_itemsets_to_save["itemsets"].apply(
    lambda x: ", ".join(sorted(map(str, x)))
)

frequent_itemsets_to_save.to_csv(
    "frequent_itemsets.csv",
    index=False
)

# Save association rules
rules_to_save = rules.copy()
rules_to_save["antecedents"] = rules_to_save["antecedents"].apply(
    lambda x: ", ".join(sorted(map(str, x)))
)
rules_to_save["consequents"] = rules_to_save["consequents"].apply(
    lambda x: ", ".join(sorted(map(str, x)))
)

rules_to_save.to_csv(
    "association_rules.csv",
    index=False
)

print("Saved:")
print("✓ frequent_itemsets.csv")
print("✓ association_rules.csv")

Saved:
✓ frequent_itemsets.csv
✓ association_rules.csv


In [10]:
saved_itemsets = pd.read_csv("frequent_itemsets.csv")
saved_rules = pd.read_csv("association_rules.csv")

print("Saved Frequent Itemsets:")
display(saved_itemsets.head(10))

print("Saved Association Rules:")
display(saved_rules.head(10))

Saved Frequent Itemsets:


,support,itemsets
0,0.0544,Biryani
1,0.1064,Brownie
2,0.0490,Burger
3,0.0526,Chole Bhature
4,0.0544,Coffee
5,0.2018,Coke
6,0.0436,Cold Coffee
7,0.0200,Donut
8,0.0952,Filter Coffee
9,0.0490,Fries


Saved Association Rules:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,Biryani,Gulab Jamun,0.0544,0.0544,0.0544,1.000000,18.382353,1.0,0.051441,inf,1.000000,1.000000,1.000000,1.000000
1,Gulab Jamun,Biryani,0.0544,0.0544,0.0544,1.000000,18.382353,1.0,0.051441,inf,1.000000,1.000000,1.000000,1.000000
2,Biryani,Raita,0.0544,0.0544,0.0544,1.000000,18.382353,1.0,0.051441,inf,1.000000,1.000000,1.000000,1.000000
3,Raita,Biryani,0.0544,0.0544,0.0544,1.000000,18.382353,1.0,0.051441,inf,1.000000,1.000000,1.000000,1.000000
4,Coffee,Brownie,0.0544,0.1064,0.0544,1.000000,9.398496,1.0,0.048612,inf,0.945008,0.511278,1.000000,0.755639
5,Brownie,Coffee,0.1064,0.0544,0.0544,0.511278,9.398496,1.0,0.048612,1.934843,1.000000,0.511278,0.483162,0.755639
6,Ice Cream,Brownie,0.0520,0.1064,0.0520,1.000000,9.398496,1.0,0.046467,inf,0.942616,0.488722,1.000000,0.744361
7,Brownie,Ice Cream,0.1064,0.0520,0.0520,0.488722,9.398496,1.0,0.046467,1.854176,1.000000,0.488722,0.460677,0.744361
8,Burger,Coke,0.0490,0.2018,0.0490,1.000000,4.955401,1.0,0.039112,inf,0.839327,0.242815,1.000000,0.621407
9,Coke,Burger,0.2018,0.0490,0.0490,0.242815,4.955401,1.0,0.039112,1.255967,1.000000,0.242815,0.203801,0.621407


In [11]:
# Demonstration of the retraining idea

orders_file = "customer_orders.csv"
training_file = "Apriori_500_Realistic_Transactions.csv"

if Path(orders_file).exists():
    orders = pd.read_csv(orders_file)
    training = pd.read_csv(training_file)

    print("Customer orders:", len(orders))
    print("Current training transactions:", len(training))

    # In the actual app, last_trained_order.txt is used
    # to select only orders that have not been added previously.
else:
    print("customer_orders.csv is not available yet.")
    print("Place an order in the Streamlit app first.")

Customer orders: 4
Current training transactions: 5000


# 🔁 Complete ML Flow

```text
Transaction Dataset
        ↓
Customer Transactions
        ↓
TransactionEncoder
        ↓
One-Hot / Boolean Matrix
        ↓
Apriori Algorithm
        ↓
Frequent Itemsets
        ↓
Association Rules
        ↓
Support + Confidence + Lift
        ↓
Recommendation Function
        ↓
Food Recommendation
        ↓
Customer Places Order
        ↓
New Order Saved
        ↓
Training Dataset Updated
        ↓
Apriori Runs Again
```

